## Spread Option Pricing Final Exam
Author: Aaro Korpela

First we start with implementing the approximation from Deng et al. (2008), proposition 6 with Geometric Brownian Motion, where q<sub>1</sub> = q<sub>2</sub> = 0

The expansion point is $y_0 = 0$. We first define the helper functions $J_0$, $J_1$, and $J_2$, followed by the main pricing function.

In [37]:
import numpy as np
from scipy.stats import norm

# Standard Normal PDF and CDF from scipyy
def n(x):
    return norm.pdf(x)

def N(x):
    return norm.cdf(x)

# --- Functions J0, J1, J2 from Equations (40), (41), (42) ---
def J0(u, v):
    return N(u / np.sqrt(1 + v**2))

def J1(u, v):
    numerator = 1 + (1 + u**2) * v**2
    denominator = (1 + v**2)**2.5
    return (numerator / denominator) * n(u / np.sqrt(1 + v**2))

def J2(u, v):
    term1 = (6 - 6 * u**2) * v**2
    term2 = (21 - 2 * u**2 - u**4) * v**4
    term3 = 4 * (3 + u**2) * v**6 - 3
    denominator = (1 + v**2)**5.5
    return ((term1 + term2 + term3) / denominator) * u * n(u / np.sqrt(1 + v**2))

# --- Main Approximation Function (Proposition 6 & Equation 50) ---
def deng_spread_option_price(S1, S2, K, r, T, sigma1, sigma2, rho):
    """
    Prices the spread option using Deng et al. closed-form approximation.
    Based on Proposition 6 and Equation (50) for the GBM case (q_i = 0).
    """
    # Equation (5) with q1 = q2 = 0
    mu1 = np.log(S1) + (r - 0.5 * sigma1**2) * T
    mu2 = np.log(S2) + (r - 0.5 * sigma2**2) * T
    nu1 = sigma1 * np.sqrt(T)
    nu2 = sigma2 * np.sqrt(T)
    
    # As stated in the paper, we set y0 = 0, page 15
    y0 = 0.0
    R = np.exp(nu2 * y0 + mu2)  # Since y0=0, R = exp(mu2)
    
    # Equations (47), (48), (49) evaluated at y0 = 0
    denom = nu1 * np.sqrt(1 - rho**2)
    C3 = (mu1 - np.log(R + K)) / denom
    D3 = (rho * nu1 - (nu2 * R) / (R + K)) / denom
    epsilon = - (nu2**2 * R * K / (R + K)**2) / (2 * denom)
    
    # Equations (43) and (44)
    C1 = C3 + D3 * rho * nu1 + epsilon * rho**2 * nu1**2 + np.sqrt(1 - rho**2) * nu1
    D1 = D3 + 2 * rho * nu1 * epsilon
    
    # Equations (45) and (46)
    C2 = C3 + D3 * nu2 + epsilon * nu2**2
    D2 = D3 + 2 * nu2 * epsilon
    
    # Equation (39): Approximated integrals I1, I2, I3 (to the second order in epsilon)
    I1 = J0(C1, D1) + J1(C1, D1) * epsilon + 0.5 * J2(C1, D1) * epsilon**2
    I2 = J0(C2, D2) + J1(C2, D2) * epsilon + 0.5 * J2(C2, D2) * epsilon**2
    I3 = J0(C3, D3) + J1(C3, D3) * epsilon + 0.5 * J2(C3, D3) * epsilon**2
    
    # Equation (50): Spread Option Price for GBMs
    price = S1 * I1 - S2 * I2 - K * np.exp(-r * T) * I3
    
    return price

We now test our implementation using the parameters provided in the exam and take K=10 to test:
* Initial prices: $S_1 = 90$, $S_2 = 100$
* Strike price: $K = 10$
* Risk-free rate: $r = 0$
* Time to maturity: $T = 0.25$
* Volatilities: $\sigma_1 = 0.4$, $\sigma_2 = 0.5$
* Correlation: $\rho = 0.8$

In [38]:
# --- Exam variables, note K ---
S1_exam = 90.0
S2_exam = 100.0
K_exam = 10.0
r_exam = 0.0
T_exam = 0.25
sigma1_exam = 0.4
sigma2_exam = 0.5
rho_exam = 0.8

# Calculate the price for K 10
approx_price = deng_spread_option_price(
    S1=S1_exam, 
    S2=S2_exam, 
    K=K_exam, 
    r=r_exam, 
    T=T_exam, 
    sigma1=sigma1_exam, 
    sigma2=sigma2_exam, 
    rho=rho_exam
)

print(f"Spread Option Price (Deng et al. Approx, K=10): {approx_price:.6f}")

Spread Option Price (Deng et al. Approx, K=10): 0.427649


To verify the numerical consistency of the Deng et al. approximation, we will compare it against two benchmarks:
1. Kirk's Approximation, which is a widely used industry standard for pricing spread options. We will compare Deng and Kirk for the $K=10$ case.
2. Margrabe's Formula, which is the exact closed-form solution for an exchange option (where the spread $K = 0$). We will verify that Deng's approximation flawlessly reduces to Margrabe's formula when $K=0$.

First, we implement Kirk's approximation and Margrabe's formula.

In [ ]:
def kirk_approximation(S1, S2, K, r, T, sigma1, sigma2, rho):
    """
    Prices the spread option using Kirk's Approximation.
    Based on Equations (93) and (94) in the paper, page 23.
    """
    # If K is 0, Kirk's approximation mathematically reduces to Margrabe
    if K == 0:
        return margrabe_formula(S1, S2, r, T, sigma1, sigma2, rho)
        
    # Adjusted S2
    S2_adj = S2 + K * np.exp(-r * T)
    
    # Equation (94): calculating sigma_K and d_K
    fraction = S2 / S2_adj
    sigma_K = np.sqrt(sigma1**2 - 2 * rho * sigma1 * sigma2 * fraction + sigma2**2 * fraction**2)
    d_K = np.log(S1 / S2_adj) / (sigma_K * np.sqrt(T))
    
    # Equation (93): Kirk's price
    d1 = d_K + 0.5 * sigma_K * np.sqrt(T)
    d2 = d_K - 0.5 * sigma_K * np.sqrt(T)
    
    price = S1 * N(d1) - S2_adj * N(d2)
    return price

def margrabe_formula(S1, S2, r, T, sigma1, sigma2, rho):
    """
    Exact closed-form price for an Exchange Option (K=0).
    Based on Equation (20) simplified for GBM with q1=q2=0, covered in the theoretical consistency with the attached pdf.
    """
    # Variance of the spread
    sigma = np.sqrt(sigma1**2 + sigma2**2 - 2 * rho * sigma1 * sigma2)
    
    d1 = (np.log(S1 / S2) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    price = S1 * N(d1) - S2 * N(d2)
    return price

In [40]:
print("--- Numerical Consistency Check ---")

# 1. Compare Deng vs Kirk for K = 10
price_kirk_10 = kirk_approximation(
    S1=S1_exam, S2=S2_exam, K=K_exam, r=r_exam, 
    T=T_exam, sigma1=sigma1_exam, sigma2=sigma2_exam, rho=rho_exam
)

print(f"K=10 | Deng Approximation: {approx_price:.6f}")
print(f"K=10 | Kirk Approximation: {price_kirk_10:.6f}")
print(f"K=10 | Absolute Diff:      {abs(approx_price - price_kirk_10):.6e}\n")

# 2. Compare Deng vs Margrabe for K = 0
K_zero = 0.0

price_deng_0 = deng_spread_option_price(
    S1=S1_exam, S2=S2_exam, K=K_zero, r=r_exam, 
    T=T_exam, sigma1=sigma1_exam, sigma2=sigma2_exam, rho=rho_exam
)

price_marg_0 = margrabe_formula(
    S1=S1_exam, S2=S2_exam, r=r_exam, 
    T=T_exam, sigma1=sigma1_exam, sigma2=sigma2_exam, rho=rho_exam
)

print(f"K=0  | Deng Approximation: {price_deng_0:.6f}")
print(f"K=0  | Margrabe Formula:   {price_marg_0:.6f}")
print(f"K=0  | Absolute Diff:      {abs(price_deng_0 - price_marg_0):.6e}")

--- Numerical Consistency Check ---
K=10 | Deng Approximation: 0.427649
K=10 | Kirk Approximation: 0.439742
K=10 | Absolute Diff:      1.209267e-02

K=0  | Deng Approximation: 2.021727
K=0  | Margrabe Formula:   2.021727
K=0  | Absolute Diff:      0.000000e+00


The numerical tests confirm the theoretical expectations outlined in the paper:
1. Comparison with Kirk's Approximation ($K=10$): The Deng approximation yields a price of $\approx 0.4276$, while Kirk gives $\approx 0.4397$. The absolute difference is small ($\approx 0.012$), which is expected. Kirk's method is known to have inherent biases for general spread options, and Deng's higher-order approximation aims to provide a more accurate valuation.
2. Reduction to Margrabe's Formula ($K=0$): When the strike price $K$ is set to $0$, the Deng approximation yields the same result as the analytical Margrabe formula (absolute difference is $0.000000e+00$). This verifies the theoretical claim that the approximation mathematically collapses into Margrabe when $K=0$.

-------

The final part of the project requires the implementation of the option sensitivities (Greeks), specifically defined in Proposition 8, Equations (51)-(58). 

Because the paper expresses these Greeks as one-dimensional integrals over a standard normal density $n(y)$, we will use Python's `scipy.integrate.quad` to numerically evaluate them from $-\infty$ to $\infty$. Furthermore, we will computationally verify the theoretical bounds for $\Delta_1$, $\Delta_2$, and $\kappa$ as stated by the authors in Equation (54) on page 17.

In [ ]:
from scipy.integrate import quad

def calculate_exact_sensitivities(S1, S2, K, r, T, sigma1, sigma2, rho):
    """
    Compute exact sensitivities Greeks using 1D numerical integration as defined in Proposition 8, Equations (51)-(58).
    """
    # Derived parameters
    mu1 = np.log(S1) + (r - 0.5 * sigma1**2) * T
    mu2 = np.log(S2) + (r - 0.5 * sigma2**2) * T
    nu1 = sigma1 * np.sqrt(T)
    nu2 = sigma2 * np.sqrt(T)
    
    # Equation (10): Exercise boundary x_bar(y)
    def x_bar(y):
        return (np.log(np.exp(nu2 * y + mu2) + K) - mu1) / nu1

    # Equation (17): Conditional moneyness A(y)
    def A_func(y):
        return (rho * y - x_bar(y)) / np.sqrt(1 - rho**2)

    # --- EQUATION 51: Delta 1 ---
    def integrand_delta1(y):
        term = A_func(y + rho * nu1) + np.sqrt(1 - rho**2) * nu1
        return N(term) * n(y)
    
    delta1, _ = quad(integrand_delta1, -np.inf, np.inf)

    # --- EQUATION 52: Delta 2 ---
    def integrand_delta2(y):
        return N(A_func(y + nu2)) * n(y)
    
    delta2 = - quad(integrand_delta2, -np.inf, np.inf)[0]

    # --- EQUATION 53: Kappa (dPi / dK) ---
    def integrand_kappa(y):
        return N(A_func(y)) * n(y)
    
    kappa = -np.exp(-r * T) * quad(integrand_kappa, -np.inf, np.inf)[0]

    # --- EQUATION 57: Vega 1 (dPi / dSigma1) ---
    def integrand_vega1_part1(y):
        return n(A_func(y)) * (K + np.exp(mu2 + nu2 * y)) * n(y)
    
    def integrand_vega1_part2(y):
        term = A_func(y + rho * nu1) + np.sqrt(1 - rho**2) * nu1
        return y * N(term) * n(y)
    
    v1_term1 = np.exp(-r * T) * np.sqrt(1 - rho**2) * np.sqrt(T) * quad(integrand_vega1_part1, -np.inf, np.inf)[0]
    v1_term2 = rho * S1 * np.sqrt(T) * quad(integrand_vega1_part2, -np.inf, np.inf)[0]
    vega1 = v1_term1 + v1_term2

    # --- EQUATION 58: Vega 2 (dPi / dSigma2) ---
    def integrand_vega2(y):
        return y * N(A_func(y + nu2)) * n(y)
    
    vega2 = -S2 * np.sqrt(T) * quad(integrand_vega2, -np.inf, np.inf)[0]

    return {
        "Delta 1 (Eq 51)": delta1,
        "Delta 2 (Eq 52)": delta2,
        "Kappa (Eq 53)": kappa,
        "Vega 1 (Eq 57)": vega1,
        "Vega 2 (Eq 58)": vega2
    }

# Execute Sensitivities for Exam Parameters (K=10)
print("--- Exact Sensitivities (Equations 51-58) for K=10 ---")
greeks = calculate_exact_sensitivities(S1_exam, S2_exam, K_exam, r_exam, T_exam, sigma1_exam, sigma2_exam, rho_exam)

for greek_name, value in greeks.items():
    print(f"{greek_name:<16}: {value:>10.6f}")

# Verification of signs based on Equation (54)
print("\n--- Equation (54) Bounds Verification ---")
print(f"0 < Delta 1 < 1     : {0 < greeks['Delta 1 (Eq 51)'] < 1}")
print(f"-1 < Delta 2 < 0    : {-1 < greeks['Delta 2 (Eq 52)'] < 0}")
print(f"-e^(-rT) < Kappa < 0: {-np.exp(-r_exam * T_exam) < greeks['Kappa (Eq 53)'] < 0}")

--- Exact Sensitivities (Equations 51-58) for K=10 ---
Delta 1 (Eq 51) :   0.081453
Delta 2 (Eq 52) :  -0.061359
Kappa (Eq 53)   :  -0.076731
Vega 1 (Eq 57)  :   1.064781
Vega 2 (Eq 58)  :   2.833531

--- Equation (54) Bounds Verification ---
0 < Delta 1 < 1     : True
-1 < Delta 2 < 0    : True
-e^(-rT) < Kappa < 0: True
